# Promise

学习目标：能用 Promise 连接异步结果、选择组合方法，并追踪拒绝传播。

前置知识：函数回调、返回值、异常处理、数组与 ES 模块。

适用版本：ECMAScript 2025、Node.js 24.11.0；.mjs 使用 ES 模块。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/javascript。

配套脚本：位于 scripts/21-promises/。

1. [states.mjs](scripts/21-promises/states.mjs)：执行器、结果采用与 thenable。
2. [chains.mjs](scripts/21-promises/chains.mjs)：返回值、错误恢复与收尾异常。
3. [combine.mjs](scripts/21-promises/combine.mjs)：四种组合、空输入与启动时机。
4. [try.mjs](scripts/21-promises/try.mjs)：同步调用转 Promise。
5. [unhandled.mjs](scripts/21-promises/unhandled.mjs)：独立的未处理拒绝反例。

Step 1：运行状态与结果示例。

```bash
node scripts/21-promises/states.mjs
```

Step 2：运行链式调用示例。

```bash
node scripts/21-promises/chains.mjs
```

Step 3：运行组合方法示例。

```bash
node scripts/21-promises/combine.mjs
```

Step 4：运行统一调用示例。

```bash
node scripts/21-promises/try.mjs
```

Step 5：单独运行预期失败的拒绝示例。

```bash
node --unhandled-rejections=strict scripts/21-promises/unhandled.mjs
```

## 1 状态、执行器与结果采用

Promise 保存一次计算的最终结果。pending 是待定，fulfilled 是已兑现，rejected 是已拒绝；后两种合称 settled（已敲定）。结果可以是任意值，拒绝原因也不限定类型，但使用 Error 能保留错误上下文。

new Promise 的执行器同步调用；resolve 与 reject 首次生效后，后续调用不能改变决定。执行器的返回值不参与兑现，执行器抛出的异常会尝试拒绝 Promise。resolve 接收另一个待定 Promise 时会采用它的结果，因此 resolved（已解决）不等于 fulfilled。

thenable 指具有可调用 then 属性的对象。resolve 会读取 then，并安排作业调用它；读取属性本身仍可能同步产生副作用或抛错。不要把不受控的 thenable 当成普通数据。

配套 [states.mjs](scripts/21-promises/states.mjs)：

```javascript
const events = [];
const inner = Promise.withResolvers();
const outer = new Promise((resolve, reject) => {
  events.push("executor");
  resolve(inner.promise);
  reject(new Error("ignored"));
  events.push("after resolve");
});
outer.then((value) => {
  events.push(`value=${value}`);
  console.log(events.join(" | ")); // → executor | after resolve | sync end | value=7
});
events.push("sync end");
inner.resolve(7);

const thenable = {
  then(resolve, reject) {
    resolve(9);
    reject(new Error("ignored again"));
  },
};
Promise.resolve(thenable).then((value) => console.log("thenable", value)); // → thenable 9
new Promise(() => { throw new Error("executor failed"); })
  .catch((error) => console.log(error.message)); // → executor failed
```

## 2 链式调用与拒绝传播

then 每次返回一个新 Promise。处理函数返回普通值时，新 Promise 以该值兑现；返回 Promise 或 thenable 时采用其结果；抛错时拒绝。不写 return 等于返回 undefined，链条无法等待一个未返回的异步任务。

缺省的成功处理会传递值，缺省的失败处理会继续传播原因。catch 是只提供拒绝处理的 then；catch 返回普通值表示恢复成功。then 的第二参数只处理上游拒绝，不能接住同一次 then 的成功回调抛出的错误，应在后面继续 catch。

finally 不接收结果参数，用来执行收尾。正常返回的值不会替换原结果，但抛错或返回拒绝的 Promise 会使后续链改为该拒绝；返回待定 Promise 会使链等待收尾完成。

配套 [chains.mjs](scripts/21-promises/chains.mjs)：

```javascript
Promise.resolve(2)
  .then((value) => value * 3)
  .then((value) => Promise.resolve(value + 1))
  .finally(() => 999)
  .then((value) => console.log("chain", value)); // → chain 7

Promise.resolve("input")
  .then(() => { throw new RangeError("callback failed"); }, () => "not reached")
  .catch((error) => {
    console.log(error.name, error.message); // → RangeError callback failed
    return "recovered";
  })
  .then((value) => console.log(value)); // → recovered

Promise.reject(new Error("operation failed"))
  .finally(() => { throw new Error("cleanup failed"); })
  .catch((error) => console.log(error.message)); // → cleanup failed

Promise.resolve(3).then(() => {}).then((value) => {
  console.log("missing return", value === undefined); // → missing return true
});
```

## 3 组合结果与创建任务

组合方法接收可迭代对象，将其中各项按 Promise.resolve 的规则处理。它们处理结果，不会自动调用数组里的任务函数，也不会因为一项先失败就取消其他任务。要决定何时启动工作，应保留函数，等真正需要时调用。

| 方法名称 | 中文名称／含义 | 非空输入的结果规则 | 空输入 |
| --- | --- | --- | --- |
| Promise.all | 全部成功 | 全部兑现后按输入顺序返回值；任一拒绝则拒绝 | 兑现为空数组 |
| Promise.allSettled | 全部敲定 | 按输入顺序返回 status 和 value 或 reason | 兑现为空数组 |
| Promise.any | 首次成功 | 首个兑现值；全部拒绝则产生 AggregateError | 拒绝，errors 为空数组 |
| Promise.race | 首次敲定 | 首个被观察到的兑现或拒绝结果 | 一直待定 |

对于已经敲定的输入，“先”受注册处理的顺序影响；不能把数组位置解释为实际 I/O 用时。下面用显式解决函数控制顺序，不依赖网络或定时器。

配套 [combine.mjs](scripts/21-promises/combine.mjs)：

```javascript
const first = Promise.withResolvers();
const second = Promise.withResolvers();
Promise.all([first.promise, second.promise]).then((values) => {
  console.log("all", JSON.stringify(values)); // → all [1,2]
});
second.resolve(2);
first.resolve(1);

Promise.allSettled([Promise.resolve(4), Promise.reject("offline")])
  .then((values) => console.log("settled", JSON.stringify(values)));
// → settled [{"status":"fulfilled","value":4},{"status":"rejected","reason":"offline"}]
Promise.any([Promise.reject("A"), Promise.resolve("B")])
  .then((value) => console.log("any", value)); // → any B
Promise.any([Promise.reject("A"), Promise.reject("B")])
  .catch((error) => console.log(error.name, error.errors.join(","))); // → AggregateError A,B
Promise.race([Promise.reject("first"), Promise.resolve("second")])
  .catch((reason) => console.log("race", reason)); // → race first

Promise.all([]).then((values) => console.log("all empty", values.length)); // → all empty 0
Promise.allSettled([]).then((values) => console.log("settled empty", values.length)); // → settled empty 0
Promise.any([]).catch((error) => console.log("any empty", error.errors.length)); // → any empty 0
let emptyRaceFinished = false;
Promise.race([]).then(() => { emptyRaceFinished = true; });
Promise.resolve().then(() => console.log("race pending", !emptyRaceFinished)); // → race pending true
// 这里只观察一个检查点；一直待定的规则来自规范，而非有限次测量。

let starts = 0;
const task = () => { starts += 1; return Promise.resolve(8); };
Promise.all([task]).then(([value]) => {
  console.log("function kept", value === task); // → function kept true
});
console.log("before call", starts); // → before call 0
Promise.all([task()]).then(([value]) => console.log("started", starts, value)); // → started 1 8

const survivor = Promise.withResolvers();
const both = Promise.all([Promise.reject("stop result"), survivor.promise]);
both.catch((reason) => {
  console.log(reason); // → stop result
  survivor.resolve("other work completed");
});
survivor.promise.then((value) => console.log(value)); // → other work completed
```

## 4 统一调用入口与未处理拒绝

Promise.withResolvers 返回 promise、resolve、reject，适合把外部完成信号接入一条 Promise；解决函数应交给明确的生产者管理，避免永远忘记结束。

Promise.try 在当前调用中执行回调，并把普通返回值、返回的 Promise 和同步异常统一为 Promise 结果。它与 Promise.resolve().then(callback) 的调用时机不同，也不同于 Promise.resolve(callback())：后一种写法在构造调用之前就可能同步抛错。

语言提供拒绝追踪宿主接口，如何报告由宿主决定。Node.js 的 unhandledRejection 与事件循环轮次有关；本章用 --unhandled-rejections=strict 固定独立反例的失败策略。给原 Promise 添加 catch 并不能自动处理另一条从 then 派生但被丢弃的拒绝链。

配套 [try.mjs](scripts/21-promises/try.mjs)：

```javascript
const order = [];
const result = Promise.try((value) => {
  order.push("callback");
  return value * 2;
}, 5);
order.push("caller");
result.then((value) => console.log(order.join(","), value)); // → callback,caller 10
Promise.try(() => { throw new TypeError("invalid input"); })
  .catch((error) => console.log(error.name, error.message)); // → TypeError invalid input
```

配套 [unhandled.mjs](scripts/21-promises/unhandled.mjs)：

```javascript
Promise.resolve().then(() => {
  throw new Error("UNHANDLED_DEMO"); // → strict 模式下退出状态为 1，诊断包含 Error: UNHANDLED_DEMO
});
```

## 本章小结

- resolve 可以采用尚未完成的结果；状态、结果与任务启动时间要分开判断。
- return 决定链条等待什么，catch 决定如何恢复，finally 的失败仍会传播。
- 组合方法不负责取消；所有可能拒绝的派生链都应有明确归属。

## 练习

1. 把 combine.mjs 中两个 resolve 的顺序交换，核对 all 的结果仍为 [1,2]，解释它为何不按完成顺序排序。
2. 设计两个都拒绝的任务，用 any 收集原因；标准：AggregateError.errors 按输入顺序包含两个原因。
3. 将 unhandled.mjs 改为返回并捕获该链；标准：输出原错误消息，进程以 0 退出。

## 参考与引用来源

- TC39（ECMA-262 第 16 版分页版）：[§27.2 Promise Objects](https://tc39.es/ecma262/2025/multipage/control-abstraction-objects.html#sec-promise-objects)，特别是 §27.2.1.3 resolve/reject、§27.2.2 Promise Jobs、§27.2.3 执行器、§27.2.4 静态方法和 §27.2.5 链与 finally。
- Node.js 24.11.0：[unhandledRejection](https://nodejs.org/download/release/v24.11.0/docs/api/process.html#event-unhandledrejection)与 [--unhandled-rejections](https://nodejs.org/download/release/v24.11.0/docs/api/cli.html#--unhandled-rejectionsmode)：宿主拒绝报告与错误策略。